In [ ]:
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt > input.txt

In [ ]:
text = open("input.txt", 'r', encoding='utf-8').read()
text[:100], len(text)

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))

In [ ]:
stoi = {}
itos = {}
for i in range(vocab_size):
    stoi[chars[i]] = i
    itos[i] = chars[i]
encode = lambda s: [stoi[c] for c in s]
decode = lambda n: [itos[i] for i in n]
encoded = encode("hello world")
decoded = "".join(decode(encoded))
encoded, decoded

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F 

data = torch.tensor(encode(text))
data.shape, data.dtype

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
train_data.shape, val_data.shape

In [ ]:
torch.manual_seed(1337)
block_size = 8
batch_size = 32
n_embd = 32

def get_batch(is_train=True):
    data = train_data if is_train else val_data
    idx = torch.randint(0, train_data.shape[0]-block_size, (batch_size, ))
    # we want shape batch x block
    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    return (x,y)
x,y = get_batch()
x.shape, y.shape

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets):
        logits = self.emb_table(idx)

        B,T,C = logits.shape
        logits = logits.view(B*T,C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_len):
        idx = torch.tensor([idx])
        out = [idx.item()]
        for i in range(max_len):
            probs = self.emb_table(idx)
            probs = F.softmax(probs, dim=1)
            idx = torch.multinomial(probs[0], num_samples=1)
            out.append(idx.item())
        return out

bg = BigramLanguageModel()

In [ ]:
print("".join(decode(bg.generate(1, 100))))

In [ ]:
optim = torch.optim.AdamW(bg.parameters(), lr=1e-3)

In [ ]:
for steps in range(10000):
    xb, yb = get_batch()
    logits, loss = bg(xb,yb)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    optim.step()
    if steps % 1000 == 0:
        print(f"step {steps}: {loss.item()}")
        